In [ ]:
# Copyright (c) Microsoft Corporation.
# Licensed under the MIT license.

# MtReNIRS quickstart

This notebook shows the public inference workflow with synthetic fNIRS only.

- The model always encodes **10-second windows at 10 Hz**.
- The safe default is `pool="none"`, which preserves one `(C, 128)` embedding per window.
- Labels, task boundaries, and linear-probe splits are downstream metadata; MtReNIRS does not infer them.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import torch

project_root = Path.cwd().resolve()
if project_root.name == "examples":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from mtrenirs.models import MtReNIRS, Recording

torch.manual_seed(7)
device = "cpu"  # Change to "cuda" when available.
print("Project:", project_root)
print("Torch:", torch.__version__, "| device:", device)

## 1. Input contract

MtReNIRS expects run-normalized HbO/HbR at 10 Hz plus Standard 1005 coordinates in the same channel order.

| Starting data | What to do |
| --- | --- |
| Raw SNIRF intensity/OD | Install `mtrenirs-models[raw]` and use `mtrenirs.models.preprocessing.load_snirf(..., align_positions=True)`. The extra includes MNE and MtSi1005. |
| Muse S Athena XDF | Install `mtrenirs-models[muse]` and use `mtrenirs.models.adapters.muse_s_athena.load_muse_s_athena_xdf(...)`. This is an explicit optional device adapter. |
| Already processed HbO/HbR | Do not preprocess twice. Ensure 10 Hz and run normalization, then construct `Recording`. |
| Processed signal without Standard 1005 coordinates | Run MtSi1005, then attach `mapping.ch_pos_1005` and `mapping.valid`. |
| Signal without labels | Embeddings are still valid. Labels are only needed later for supervised probing. |

In [ ]:
# Query public Standard 1005 slots instead of copying coordinate values.
from mtrenirs.alignment import MtSi1005

reference = MtSi1005.load_reference("standard_1005")
slot_lookup = dict(zip(reference["labels"], reference["positions"]))
pair_positions = np.asarray(
    [slot_lookup[label] for label in ("Fp1h", "Fp2h", "AFF5h", "AFF6h")],
    dtype=np.float32,
)
ch_types = ("hbo", "hbr") * 4
ch_pos_1005 = np.repeat(pair_positions, 2, axis=0)

model = MtReNIRS.from_pretrained(
    ch_types=ch_types,
    ch_pos_1005=ch_pos_1005,
    device=device,
)
print("Window:", model.window_seconds, "s | embedding dimension:", model.embedding_dim)

## 2. Whole run: keep every 10-second embedding

For a run that may contain both rest and task, use the default `none`. It preserves time instead of averaging different states together.

The default remainder policy is non-overlapping `drop`. Use `pad` only when retaining the final partial window is more important than avoiding zero padding.

In [ ]:
# A synthetic 35-second run at 10 Hz: (C, T) = (8, 350).
run_signal = torch.randn(8, 350)

run_drop = model.embed(run_signal)  # default: pool="none", remainder="drop"
run_pad = model.embed(run_signal, remainder="pad")

print("35 s default drop:", tuple(run_drop.shape), "-> 3 full windows")
print("35 s non-overlap pad:", tuple(run_pad.shape), "-> 4 windows; last 5 s padded")

## 3. Choosing `none`, `flatten`, or `temporal_mean`

| Pool | Shape | Use it when |
| --- | --- | --- |
| `none` | `(B, W, C, 128)` | Whole runs, one label per 10-second window, or custom aggregation. This is the default. |
| `flatten` | `(B, W*C*128)` | Every sample is a fixed-duration trial and one label describes the whole trial. It preserves window order but requires the same `W` for every sample. |
| `temporal_mean` | `(B, C, 128)` | Trial durations vary but all windows in each selected segment represent the same task/state. Do not average a run that mixes rest and task. |

For exactly 10 seconds (`W=1`), temporal mean and flatten contain identical values after reshaping; only their displayed shapes differ.

In [ ]:
trial_none = model.embed(run_signal, remainder="pad")
trial_flat = model.embed(run_signal, pool="flatten", remainder="pad")
trial_mean = model.embed(run_signal, pool="temporal_mean", remainder="pad")

print("35 s none:         ", tuple(trial_none.shape))
print("35 s flatten:      ", tuple(trial_flat.shape))
print("35 s temporal mean:", tuple(trial_mean.shape))

signal_10s = run_signal[:, :100]
flat_10s = model.embed(signal_10s, pool="flatten")
mean_10s = model.embed(signal_10s, pool="temporal_mean").reshape(1, -1)
print("10 s flatten equals reshaped temporal mean:", torch.equal(flat_10s, mean_10s))

## 4. Linear probing

If labels are defined every 10 seconds, keep the run output as `none`, then flatten **each window separately**. This produces `W` supervised samples. Do not flatten the entire run into one sample.

Split train/test data by subject or run before fitting a probe; splitting neighboring windows can leak subject and run information. Exclude windows that cross a task/rest transition unless the study defines a labeling rule.

In [ ]:
# run_drop has shape (B=1, W=3, C=8, D=128).
window_embeddings = run_drop[0]
probe_features = window_embeddings.reshape(window_embeddings.shape[0], -1)

# Demonstration labels only. Real labels must come from the experiment event table.
window_labels = torch.tensor([0, 1, 0], device=device)
linear_probe = torch.nn.Linear(probe_features.shape[1], 2, device=device)
logits = linear_probe(probe_features)
loss = torch.nn.functional.cross_entropy(logits, window_labels)

print("One feature row per 10 s window:", tuple(probe_features.shape))
print("Labels:", tuple(window_labels.shape), "| logits:", tuple(logits.shape))
print("Example probe loss:", round(float(loss), 4))

## 5. Preprocessing routes

### Raw SNIRF

Install `mtrenirs-models[raw]`, then let the helper produce a normalized `Recording`. This extra includes MNE, SciPy, and MtSi1005:

```python
from mtrenirs.models import MtReNIRS
from mtrenirs.models.preprocessing import load_snirf

recording = load_snirf("recording.snirf", align_positions=True)
result = MtReNIRS.from_pretrained().embed_recording(recording)
```

This route applies the supported fNIRS chain and montage alignment. Do not use it on data that have already undergone the same processing.

### Muse S Athena XDF

Muse support is an explicit optional adapter, not generic XDF inference:

```python
from mtrenirs.models import MtReNIRS
from mtrenirs.models.adapters.muse_s_athena import load_muse_s_athena_xdf

recording = load_muse_s_athena_xdf("recording.xdf")
result = MtReNIRS.from_pretrained().embed_recording(recording)
```

Install it with `mtrenirs-models[muse]`. The file may live anywhere; the adapter identifies the `Muse-OPTICS` stream rather than depending on a BIDS folder name.

### Already processed HbO/HbR

When data are already filtered, resampled to 10 Hz, and run-normalized, construct `Recording` directly. Calling `model.embed(tensor)` also assumes preprocessing is already complete; it does not preprocess the tensor.

In [ ]:
# Synthetic stand-in for already processed, run-normalized HbO/HbR.
processed_signal = run_signal.numpy().astype(np.float32)
processed_signal -= processed_signal.mean(axis=1, keepdims=True)
processed_signal /= processed_signal.std(axis=1, keepdims=True)
processed_signal = np.clip(processed_signal, -6, 6)

recording = Recording(
    signal=processed_signal,
    sfreq=10.0,
    ch_names=tuple(f"ch{index + 1}" for index in range(8)),
    ch_types=tuple(ch_types),
    ch_pos_1005=ch_pos_1005,
    ch_pos_1005_valid=np.ones(8, dtype=bool),
    normalized=True,
)
recording_result = model.embed_recording(recording)
print("Recording output (W, C, D):", tuple(recording_result.embeddings.shape))
print("Window onsets (s):", recording_result.window_onsets_seconds.tolist())

## Practical defaults

1. **Whole run or unknown boundaries:** `embed_recording(recording)` → `(W,C,128)`.
2. **10-second labels:** reshape each window to `(C*128)` for a linear probe.
3. **Fixed-duration trial:** use explicit `flatten` when window order matters.
4. **Variable-duration single-state trial:** use `temporal_mean` when averaging is scientifically justified.
5. **Final partial window:** default `drop`; choose `pad` explicitly to retain a non-overlapping tail.

The model never invents labels or task boundaries. Keep event metadata and embedding timestamps together.